# 3_KineRepo

In [1]:
"""
WINDOW-BASED KINEMATIC PREDICTION
WITH DETAILED PERFORMANCE REPORT
============================================================

For every window:

    First 80% -> known ground truth
    Last 20%  -> predicted using constant-acceleration kinematics

Windows may have different durations and different numbers
of samples.

The kinematic model is fitted independently for X, Y, and Z:

    p(t) = p0 + v0*t + 0.5*a*t^2

The first 80% of each window is used to estimate the
constant-acceleration model.

The remaining 20% is predicted using the actual timestamps
from the dataset.

Output workbook sheets:

    1. Predicted Dataset
    2. Overall Statistics
    3. Per Window
    4. Threshold Performance
"""


from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# USER SETTINGS
# ============================================================


INPUT_FILE = Path(
    r"C:\\Users\\samiu\\Desktop\\Kinematics\\Test02_bvh_Global_RArm\\Window Creation\\bvh_Meter_Global_RightArm_windowed.xlsx"
)


OUTPUT_FILE = Path(
    r"C:\\Users\\samiu\\Desktop\\Kinematics\\Test02_bvh_Global_RArm\\Window Creation\\Kinematic Prediction\\RArm_KinematicPrediction_Report.xlsx"
)


# ------------------------------------------------------------
# Percentage of each window used as known data.
# ------------------------------------------------------------

KNOWN_FRACTION = 0.80


# ------------------------------------------------------------
# Error thresholds for threshold-performance analysis.
# ------------------------------------------------------------

ERROR_THRESHOLDS = [

    0.001,

    0.005,

    0.010,

    0.015,

    0.020,

    0.025,

    0.030,

    0.035,

    0.040,

    0.045,

    0.050,

    0.060,

    0.070,

    0.090,

    0.095,

    0.100,

    0.150,

    0.200,

    0.250,

    0.300

]


# ============================================================
# COLUMN DEFINITIONS
# ============================================================


TIME_COLUMN = "Time (s)"

WINDOW_COLUMN = "Window Number"


POSITION_COLUMNS = [

    "X Position (m)",

    "Y Position (m)",

    "Z Position (m)"

]


PREDICTED_COLUMNS = [

    "Predicted X Position (m)",

    "Predicted Y Position (m)",

    "Predicted Z Position (m)"

]


ERROR_COLUMNS = [

    "Prediction Error X (m)",

    "Prediction Error Y (m)",

    "Prediction Error Z (m)"

]


ERROR_MAGNITUDE_COLUMN = (
    "Prediction Error Magnitude (m)"
)


# ============================================================
# HELPER FUNCTIONS
# ============================================================


def print_line(character="=", length=75):

    print(
        character * length
    )


# ------------------------------------------------------------


def validate_columns(df):

    required_columns = (

        [
            TIME_COLUMN,
            WINDOW_COLUMN
        ]

        +

        POSITION_COLUMNS

    )


    missing_columns = [

        column

        for column in required_columns

        if column not in df.columns

    ]


    if missing_columns:

        raise ValueError(

            "Missing required columns:\n"

            +

            "\n".join(

                f"  - {column}"

                for column in missing_columns

            )

        )


# ------------------------------------------------------------


def fit_constant_acceleration(

    time_values,

    position_values

):

    """
    Fit:

        p(t) = p0 + v0*t + 0.5*a*t²

    using least squares.

    Time is shifted such that the first known sample
    is t = 0.
    """


    t = (

        time_values

        -

        time_values[0]

    )


    design_matrix = np.column_stack(

        [

            np.ones_like(t),

            t,

            0.5 * t ** 2

        ]

    )


    coefficients, _, _, _ = (

        np.linalg.lstsq(

            design_matrix,

            position_values,

            rcond=None

        )

    )


    p0 = coefficients[0]

    v0 = coefficients[1]

    acceleration = coefficients[2]


    return (

        p0,

        v0,

        acceleration

    )


# ------------------------------------------------------------


def predict_position(

    p0,

    v0,

    acceleration,

    delta_t

):

    """
    Constant acceleration equation:

        p = p0 + v0*t + 0.5*a*t²
    """


    return (

        p0

        +

        v0 * delta_t

        +

        0.5
        *
        acceleration
        *
        delta_t ** 2

    )


# ------------------------------------------------------------


def calculate_axis_metrics(

    actual,

    predicted

):

    """
    Calculate MAE, RMSE and R² for one coordinate axis.
    """


    error = (

        predicted

        -

        actual

    )


    mae = np.mean(

        np.abs(error)

    )


    rmse = np.sqrt(

        np.mean(

            error ** 2

        )

    )


    ss_res = np.sum(

        error ** 2

    )


    ss_tot = np.sum(

        (

            actual

            -

            np.mean(actual)

        )

        **

        2

    )


    if np.isclose(

        ss_tot,

        0

    ):

        r_squared = np.nan

    else:

        r_squared = (

            1

            -

            ss_res
            /
            ss_tot

        )


    return {

        "MAE": mae,

        "RMSE": rmse,

        "R2": r_squared

    }


# ============================================================
# START
# ============================================================


print()

print_line("=")

print(
    "WINDOW-BASED KINEMATIC PREDICTION"
)

print_line("=")


print()

print(
    "Loading dataset..."
)


# ============================================================
# LOAD DATA
# ============================================================


if not INPUT_FILE.exists():

    raise FileNotFoundError(

        f"\nInput file not found:\n"
        f"{INPUT_FILE}"

    )


df = pd.read_excel(

    INPUT_FILE

)


validate_columns(df)


# ============================================================
# CONVERT TO NUMERIC
# ============================================================


numeric_columns = (

    [
        TIME_COLUMN,
        WINDOW_COLUMN
    ]

    +

    POSITION_COLUMNS

)


for column in numeric_columns:

    df[column] = pd.to_numeric(

        df[column],

        errors="coerce"

    )


# ------------------------------------------------------------
# Check invalid values.
# ------------------------------------------------------------


if df[
    numeric_columns
].isna().any().any():

    invalid_rows = (

        df[
            numeric_columns
        ]

        .isna()

        .any(axis=1)

        .sum()

    )


    raise ValueError(

        f"Dataset contains invalid values.\n"
        f"Invalid rows: {invalid_rows}"

    )


# ============================================================
# SORT DATA
# ============================================================


df = (

    df

    .sort_values(

        TIME_COLUMN

    )

    .reset_index(

        drop=True

    )

)


# ============================================================
# INITIALIZE OUTPUT COLUMNS
# ============================================================


for column in PREDICTED_COLUMNS:

    df[column] = np.nan


for column in ERROR_COLUMNS:

    df[column] = np.nan


df[
    ERROR_MAGNITUDE_COLUMN
] = np.nan


# ============================================================
# GET WINDOWS
# ============================================================


window_numbers = (

    df[
        WINDOW_COLUMN
    ]

    .unique()

)


print(
    "Saving output file..."
)


# ============================================================
# STORAGE FOR REPORTING
# ============================================================


window_summary = []


all_prediction_indices = []


# ============================================================
# PROCESS EACH WINDOW
# ============================================================


for window_number in window_numbers:


    # --------------------------------------------------------
    # Extract current window.
    # --------------------------------------------------------


    window_df = (

        df[
            df[
                WINDOW_COLUMN
            ]

            ==

            window_number
        ]

        .sort_values(

            TIME_COLUMN

        )

        .copy()

    )


    window_indices = (

        window_df.index.to_numpy()

    )


    n_samples = len(

        window_df

    )


    # --------------------------------------------------------
    # Skip extremely small windows.
    # --------------------------------------------------------


    if n_samples < 4:

        print()

        print_line("=")

        print(

            f"WINDOW {window_number}"

        )

        print_line("=")

        print(

            f"Window contains only "
            f"{n_samples} samples."

        )

        print(

            "Too few samples for "
            "constant-acceleration fitting."

        )

        continue


    # ========================================================
    # DETERMINE 80 / 20 SPLIT
    # ========================================================


    known_samples = int(

        np.floor(

            KNOWN_FRACTION

            *

            n_samples

        )

    )


    known_samples = max(

        3,

        known_samples

    )


    if known_samples >= n_samples:

        known_samples = (

            n_samples

            -

            1

        )


    predicted_samples = (

        n_samples

        -

        known_samples

    )


    # ========================================================
    # EXTRACT TIME
    # ========================================================


    time_values = (

        window_df[
            TIME_COLUMN
        ]

        .to_numpy(

            dtype=float

        )

    )


    known_time = (

        time_values[
            :known_samples
        ]

    )


    prediction_time = (

        time_values[
            known_samples:
        ]

    )


    window_start_time = (

        time_values[0]

    )


    prediction_start_time = (

        prediction_time[0]

    )


    window_end_time = (

        time_values[-1]

    )


    # ========================================================
    # COPY GROUND TRUTH INTO FIRST 80%
    # ========================================================


    known_indices = (

        window_indices[
            :known_samples
        ]

    )


    for (

        position_column,

        predicted_column

    ) in zip(

        POSITION_COLUMNS,

        PREDICTED_COLUMNS

    ):


        df.loc[
            known_indices,
            predicted_column
        ] = (

            df.loc[
                known_indices,
                position_column
            ]

        )


    # ========================================================
    # FIT X, Y, Z KINEMATIC MODELS
    # ========================================================


    estimated_parameters = []


    for position_column in POSITION_COLUMNS:


        known_position = (

            window_df[
                position_column
            ]

            .iloc[
                :known_samples
            ]

            .to_numpy(

                dtype=float

            )

        )


        parameters = (

            fit_constant_acceleration(

                known_time,

                known_position

            )

        )


        estimated_parameters.append(

            parameters

        )


    # ========================================================
    # PREDICT FINAL 20%
    # ========================================================


    prediction_indices = (

        window_indices[
            known_samples:
        ]

    )


    all_prediction_indices.extend(

        prediction_indices.tolist()

    )


    # Actual timestamps relative to beginning of window.

    delta_t_prediction = (

        prediction_time

        -

        window_start_time

    )


    for axis_index, predicted_column in enumerate(

        PREDICTED_COLUMNS

    ):


        p0 = (

            estimated_parameters[
                axis_index
            ][0]

        )


        v0 = (

            estimated_parameters[
                axis_index
            ][1]

        )


        acceleration = (

            estimated_parameters[
                axis_index
            ][2]

        )


        predicted_position = (

            predict_position(

                p0,

                v0,

                acceleration,

                delta_t_prediction

            )

        )


        df.loc[
            prediction_indices,
            predicted_column
        ] = (

            predicted_position

        )


    # ========================================================
    # CALCULATE ERRORS
    # ========================================================


    for (

        position_column,

        predicted_column,

        error_column

    ) in zip(

        POSITION_COLUMNS,

        PREDICTED_COLUMNS,

        ERROR_COLUMNS

    ):


        df.loc[
            window_indices,
            error_column
        ] = (

            df.loc[
                window_indices,
                predicted_column
            ]

            -

            df.loc[
                window_indices,
                position_column
            ]

        )


    # --------------------------------------------------------
    # 3D error magnitude.
    # --------------------------------------------------------


    error_x = (

        df.loc[
            window_indices,
            ERROR_COLUMNS[0]
        ]

        .to_numpy()

    )


    error_y = (

        df.loc[
            window_indices,
            ERROR_COLUMNS[1]
        ]

        .to_numpy()

    )


    error_z = (

        df.loc[
            window_indices,
            ERROR_COLUMNS[2]
        ]

        .to_numpy()

    )


    error_magnitude = np.sqrt(

        error_x ** 2

        +

        error_y ** 2

        +

        error_z ** 2

    )


    df.loc[
        window_indices,
        ERROR_MAGNITUDE_COLUMN
    ] = (

        error_magnitude

    )


    # ========================================================
    # WINDOW PERFORMANCE
    # ========================================================


    prediction_error_x = (

        error_x[
            known_samples:
        ]

    )


    prediction_error_y = (

        error_y[
            known_samples:
        ]

    )


    prediction_error_z = (

        error_z[
            known_samples:
        ]

    )


    prediction_error_magnitude = (

        error_magnitude[
            known_samples:
        ]

    )


    # Axis MAE.

    x_mae = np.mean(

        np.abs(

            prediction_error_x

        )

    )


    y_mae = np.mean(

        np.abs(

            prediction_error_y

        )

    )


    z_mae = np.mean(

        np.abs(

            prediction_error_z

        )

    )


    # Axis RMSE.

    x_rmse = np.sqrt(

        np.mean(

            prediction_error_x ** 2

        )

    )


    y_rmse = np.sqrt(

        np.mean(

            prediction_error_y ** 2

        )

    )


    z_rmse = np.sqrt(

        np.mean(

            prediction_error_z ** 2

        )

    )


    # 3D metrics.

    window_3d_mae = np.mean(

        prediction_error_magnitude

    )


    window_3d_rmse = np.sqrt(

        np.mean(

            prediction_error_magnitude

            **

            2

        )

    )


    window_max_error = np.max(

        prediction_error_magnitude

    )


    # ========================================================
    # SAVE WINDOW SUMMARY
    # ========================================================


    window_summary.append({

        "Window Number":

            window_number,


        "Total Samples":

            n_samples,


        "Known 80% Samples":

            known_samples,


        "Predicted 20% Samples":

            predicted_samples,


        "Window Start (s)":

            window_start_time,


        "Prediction Starts (s)":

            prediction_start_time,


        "Window End (s)":

            window_end_time,


        "Estimated Acc X (m/s^2)":

            estimated_parameters[0][2],


        "Estimated Acc Y (m/s^2)":

            estimated_parameters[1][2],


        "Estimated Acc Z (m/s^2)":

            estimated_parameters[2][2],


        "X MAE (m)":

            x_mae,


        "Y MAE (m)":

            y_mae,


        "Z MAE (m)":

            z_mae,


        "X RMSE (m)":

            x_rmse,


        "Y RMSE (m)":

            y_rmse,


        "Z RMSE (m)":

            z_rmse,


        "3D MAE (m)":

            window_3d_mae,


        "3D RMSE (m)":

            window_3d_rmse,


        "Maximum 3D Error (m)":

            window_max_error

    })


    # ========================================================
    # PRINT WINDOW REPORT
    # ========================================================


    print()

    print_line("=")

    print(

        f"WINDOW {window_number}"

    )

    print_line("=")


    print(

        f"Total samples        : "
        f"{n_samples}"

    )


    print(

        f"Known 80% samples    : "
        f"{known_samples}"

    )


    print(

        f"Predicted 20% samples: "
        f"{predicted_samples}"

    )


    print(

        f"Window start         : "
        f"{window_start_time:.6f} s"

    )


    print(

        f"Prediction starts    : "
        f"{prediction_start_time:.6f} s"

    )


    print(

        f"Window end           : "
        f"{window_end_time:.6f} s"

    )


    print()


    print(

        f"Estimated Acc X      : "
        f"{estimated_parameters[0][2]:.8f} m/s²"

    )


    print(

        f"Estimated Acc Y      : "
        f"{estimated_parameters[1][2]:.8f} m/s²"

    )


    print(

        f"Estimated Acc Z      : "
        f"{estimated_parameters[2][2]:.8f} m/s²"

    )


    print()


    print(

        f"3D MAE               : "
        f"{window_3d_mae:.10f} m"

    )


    print(

        f"3D RMSE              : "
        f"{window_3d_rmse:.10f} m"

    )


    print(

        f"Maximum error        : "
        f"{window_max_error:.10f} m"

    )


# ============================================================
# CONVERT SUMMARY TO DATAFRAME
# ============================================================


window_summary_df = pd.DataFrame(

    window_summary

)


# ============================================================
# EXTRACT ACTUAL PREDICTION DATA
# ============================================================


prediction_df = (

    df.loc[
        all_prediction_indices
    ]

    .copy()

)


prediction_df = (

    prediction_df

    .sort_values(

        TIME_COLUMN

    )

)


# ============================================================
# AXIS-WISE METRICS
# ============================================================


actual_x = (

    prediction_df[
        POSITION_COLUMNS[0]
    ]

    .to_numpy()

)


actual_y = (

    prediction_df[
        POSITION_COLUMNS[1]
    ]

    .to_numpy()

)


actual_z = (

    prediction_df[
        POSITION_COLUMNS[2]
    ]

    .to_numpy()

)


predicted_x = (

    prediction_df[
        PREDICTED_COLUMNS[0]
    ]

    .to_numpy()

)


predicted_y = (

    prediction_df[
        PREDICTED_COLUMNS[1]
    ]

    .to_numpy()

)


predicted_z = (

    prediction_df[
        PREDICTED_COLUMNS[2]
    ]

    .to_numpy()

)


metrics_x = calculate_axis_metrics(

    actual_x,

    predicted_x

)


metrics_y = calculate_axis_metrics(

    actual_y,

    predicted_y

)


metrics_z = calculate_axis_metrics(

    actual_z,

    predicted_z

)


# ============================================================
# 3D ERROR METRICS
# ============================================================


prediction_error_magnitude = (

    prediction_df[
        ERROR_MAGNITUDE_COLUMN
    ]

    .to_numpy()

)


overall_3d_mae = np.mean(

    prediction_error_magnitude

)


overall_3d_rmse = np.sqrt(

    np.mean(

        prediction_error_magnitude

        **

        2

    )

)


overall_max_error = np.max(

    prediction_error_magnitude

)


overall_error_std = np.std(

    prediction_error_magnitude

)


# ============================================================
# ERROR PERCENTILES
# ============================================================


percentiles = {

    "50th":

        np.percentile(

            prediction_error_magnitude,

            50

        ),


    "75th":

        np.percentile(

            prediction_error_magnitude,

            75

        ),


    "90th":

        np.percentile(

            prediction_error_magnitude,

            90

        ),


    "95th":

        np.percentile(

            prediction_error_magnitude,

            95

        ),


    "99th":

        np.percentile(

            prediction_error_magnitude,

            99

        )

}


# ============================================================
# THRESHOLD PERFORMANCE
# ============================================================


threshold_results = []


total_prediction_samples = len(

    prediction_error_magnitude

)


for threshold in ERROR_THRESHOLDS:


    count = np.sum(

        prediction_error_magnitude

        <=

        threshold

    )


    percentage = (

        100

        *

        count

        /

        total_prediction_samples

    )


    threshold_results.append({

        "Error Threshold (m)":

            threshold,


        "Samples Within Threshold":

            count,


        "Percentage (%)":

            percentage

    })


threshold_df = pd.DataFrame(

    threshold_results

)


# ============================================================
# BEST AND WORST WINDOW
# ============================================================


best_window_row = (

    window_summary_df.loc[

        window_summary_df[
            "3D RMSE (m)"
        ].idxmin()

    ]

)


worst_window_row = (

    window_summary_df.loc[

        window_summary_df[
            "3D RMSE (m)"
        ].idxmax()

    ]

)


# ============================================================
# GENERAL STATISTICS
# ============================================================


total_samples = len(

    df

)


number_of_windows = len(

    window_summary_df

)


prediction_percentage = (

    100

    *

    total_prediction_samples

    /

    total_samples

)


# ============================================================
# OVERALL STATISTICS DATAFRAME
# ============================================================


overall_statistics = [

    ["Number of Windows", number_of_windows],

    ["Total Samples", total_samples],

    ["Prediction Samples", total_prediction_samples],

    ["Prediction Percentage (%)", prediction_percentage],


    ["X MAE (m)", metrics_x["MAE"]],

    ["Y MAE (m)", metrics_y["MAE"]],

    ["Z MAE (m)", metrics_z["MAE"]],


    ["X RMSE (m)", metrics_x["RMSE"]],

    ["Y RMSE (m)", metrics_y["RMSE"]],

    ["Z RMSE (m)", metrics_z["RMSE"]],


    ["3D MAE (m)", overall_3d_mae],

    ["3D RMSE (m)", overall_3d_rmse],

    ["Maximum 3D Error (m)", overall_max_error],

    ["3D Error Standard Deviation (m)", overall_error_std],


    ["50th Percentile (m)", percentiles["50th"]],

    ["75th Percentile (m)", percentiles["75th"]],

    ["90th Percentile (m)", percentiles["90th"]],

    ["95th Percentile (m)", percentiles["95th"]],

    ["99th Percentile (m)", percentiles["99th"]],


    ["R² X", metrics_x["R2"]],

    ["R² Y", metrics_y["R2"]],

    ["R² Z", metrics_z["R2"]],


    ["Best Window", best_window_row["Window Number"]],

    ["Best Window 3D RMSE (m)", best_window_row["3D RMSE (m)"]],

    ["Worst Window", worst_window_row["Window Number"]],

    ["Worst Window 3D RMSE (m)", worst_window_row["3D RMSE (m)"]]

]


overall_statistics_df = pd.DataFrame(

    overall_statistics,

    columns=[

        "Metric",

        "Value"

    ]

)


# ============================================================
# PRINT OVERALL REPORT
# ============================================================


print()

print_line("=")

print(
    "OVERALL PERFORMANCE"
)

print_line("=")


# ------------------------------------------------------------
# GENERAL
# ------------------------------------------------------------


print()

print(
    "GENERAL"
)

print_line("-")


print(

    f"Number of windows       : "
    f"{number_of_windows}"

)


print(

    f"Total samples           : "
    f"{total_samples}"

)


print(

    f"Prediction samples      : "
    f"{total_prediction_samples}"

)


print(

    f"Prediction percentage   : "
    f"{prediction_percentage:.2f}%"

)


# ------------------------------------------------------------
# AXIS-WISE MAE
# ------------------------------------------------------------


print()

print(
    "AXIS-WISE MAE"
)

print_line("-")


print(

    f"X MAE                   : "
    f"{metrics_x['MAE']:.10f} m"

)


print(

    f"Y MAE                   : "
    f"{metrics_y['MAE']:.10f} m"

)


print(

    f"Z MAE                   : "
    f"{metrics_z['MAE']:.10f} m"

)


# ------------------------------------------------------------
# AXIS-WISE RMSE
# ------------------------------------------------------------


print()

print(
    "AXIS-WISE RMSE"
)

print_line("-")


print(

    f"X RMSE                  : "
    f"{metrics_x['RMSE']:.10f} m"

)


print(

    f"Y RMSE                  : "
    f"{metrics_y['RMSE']:.10f} m"

)


print(

    f"Z RMSE                  : "
    f"{metrics_z['RMSE']:.10f} m"

)


# ------------------------------------------------------------
# 3D ERROR
# ------------------------------------------------------------


print()

print(
    "3D POSITION ERROR"
)

print_line("-")


print(

    f"3D MAE                  : "
    f"{overall_3d_mae:.10f} m"

)


print(

    f"3D RMSE                 : "
    f"{overall_3d_rmse:.10f} m"

)


print(

    f"Maximum 3D error        : "
    f"{overall_max_error:.10f} m"

)


print(

    f"3D error std            : "
    f"{overall_error_std:.10f} m"

)


# ------------------------------------------------------------
# PERCENTILES
# ------------------------------------------------------------


print()

print(
    "ERROR PERCENTILES"
)

print_line("-")


for percentile_name, value in percentiles.items():

    print(

        f"{percentile_name} percentile"
        f"{' ' * (9 - len(percentile_name))}"
        f": {value:.10f} m"

    )


# ------------------------------------------------------------
# R²
# ------------------------------------------------------------


print()

print(
    "R²"
)

print_line("-")


print(

    f"R² X                    : "
    f"{metrics_x['R2']:.8f}"

)


print(

    f"R² Y                    : "
    f"{metrics_y['R2']:.8f}"

)


print(

    f"R² Z                    : "
    f"{metrics_z['R2']:.8f}"

)


# ------------------------------------------------------------
# THRESHOLD PERFORMANCE
# ------------------------------------------------------------


print()

print(
    "THRESHOLD PERFORMANCE"
)

print_line("-")


for result in threshold_results:


    threshold = (

        result[
            "Error Threshold (m)"
        ]

    )


    count = (

        result[
            "Samples Within Threshold"
        ]

    )


    percentage = (

        result[
            "Percentage (%)"
        ]

    )


    print(

        f"Error <= {threshold:.3f} m"
        f"     : "
        f"{percentage:8.3f}% "
        f"({count} samples)"

    )


# ------------------------------------------------------------
# BEST / WORST WINDOW
# ------------------------------------------------------------


print()

print(
    "BEST / WORST WINDOW"
)

print_line("-")


print(

    f"Best window             : "
    f"{best_window_row['Window Number']}"

)


print(

    f"Best window RMSE        : "
    f"{best_window_row['3D RMSE (m)']:.10f} m"

)


print()

print(

    f"Worst window            : "
    f"{worst_window_row['Window Number']}"

)


print(

    f"Worst window RMSE       : "
    f"{worst_window_row['3D RMSE (m)']:.10f} m"

)


print()

print_line("=")


# ============================================================
# SAVE EXCEL FILE
# ============================================================


print()

print(
    "Saving Excel workbook..."
)


with pd.ExcelWriter(

    OUTPUT_FILE,

    engine="openpyxl"

) as writer:


    # --------------------------------------------------------
    # Sheet 1
    # --------------------------------------------------------

    df.to_excel(

        writer,

        sheet_name="Predicted Dataset",

        index=False

    )


    # --------------------------------------------------------
    # Sheet 2
    # --------------------------------------------------------

    overall_statistics_df.to_excel(

        writer,

        sheet_name="Overall Statistics",

        index=False

    )


    # --------------------------------------------------------
    # Sheet 3
    # --------------------------------------------------------

    window_summary_df.to_excel(

        writer,

        sheet_name="Per Window",

        index=False

    )


    # --------------------------------------------------------
    # Sheet 4
    # --------------------------------------------------------

    threshold_df.to_excel(

        writer,

        sheet_name="Threshold Performance",

        index=False

    )


# ============================================================
# FINAL REPORT
# ============================================================


print()

print_line("=")

print(
    "REPORT SAVED SUCCESSFULLY"
)

print_line("=")


print()

print(
    f"Output file:\n"
    f"{OUTPUT_FILE}"
)


print()

print(
    "The output workbook contains:"
)


print()

print(
    "  1. Predicted Dataset"
)

print(
    "  2. Overall Statistics"
)

print(
    "  3. Per Window"
)

print(
    "  4. Threshold Performance"
)


print()

print_line("=")


WINDOW-BASED KINEMATIC PREDICTION

Loading dataset...
Saving output file...

WINDOW 1
Total samples        : 58
Known 80% samples    : 46
Predicted 20% samples: 12
Window start         : 0.000000 s
Prediction starts    : 0.386451 s
Window end           : 0.478863 s

Estimated Acc X      : -1.63446911 m/s²
Estimated Acc Y      : -0.09526998 m/s²
Estimated Acc Z      : -0.19004125 m/s²

3D MAE               : 0.0077945990 m
3D RMSE              : 0.0083104969 m
Maximum error        : 0.0123824629 m

WINDOW 2
Total samples        : 58
Known 80% samples    : 46
Predicted 20% samples: 12
Window start         : 0.487265 s
Prediction starts    : 0.873716 s
Window end           : 0.966128 s

Estimated Acc X      : 0.14391371 m/s²
Estimated Acc Y      : -0.05221030 m/s²
Estimated Acc Z      : 0.89968059 m/s²

3D MAE               : 0.0261707985 m
3D RMSE              : 0.0276464696 m
Maximum error        : 0.0381716292 m

WINDOW 3
Total samples        : 58
Known 80% samples    : 46
Predicted 2